#### Companies House signals: director changes, filings, financial health

All signals come from Companies House and key on CompanyNumber.
This notebook builds two relationship signals for existing-client work, both from
live Companies House APIs:

1. Director changes - directors appointed or resigned recently (Officers API).
2. Growth filings - recent share allotments, filing type SH01, which can indicate a
   capital raise (Filing History API).


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import requests
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv

load_dotenv()

CH_API_KEY = os.getenv("COMPANIES_HOUSE_API_KEY")


In [3]:
data_path = r"C:\MSC\Project\lloyds-commercial-banking-intelligence-2026\data\processed\filtered_bb_sme_sectors.csv"
df = pd.read_csv(data_path)
df.columns = df.columns.str.strip()

# CompanyNumber is the shared join key, zero-padded to 8 digits.
df["CompanyNumber"] = df["CompanyNumber"].astype(str).str.zfill(8)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df[["CompanyName", "CompanyNumber", "CompanyStatus"]].head()

C:\conda_temp\ipykernel_17044\2829048442.py:2: DtypeWarning: Columns (43,44,45,46,47,48,49,50,51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


Rows: 1372321
Columns: 57


,CompanyName,CompanyNumber,CompanyStatus
0,!ABRIDGE TAX LTD,16092999,Active
1,!BIG IMPACT GRAPHICS LIMITED,11743365,Active
2,!NFLECTION ADVISORY LIMITED,15073164,Active
3,!NFOGENIE LTD,13522064,Active
4,!NNOV8 LIMITED,11006939,Active


In [13]:
API_LIMIT = 1000

base = df[df["CompanyStatus"].astype(str).str.lower() == "active"].copy()

# Priority flag: companies with an outstanding charge go to the front of the queue.
outstanding = pd.to_numeric(base.get("Mortgages.NumMortOutstanding"), errors="coerce").fillna(0)
base["_priority"] = (outstanding > 0).astype(int)

subset = (
    base.sort_values("_priority", ascending=False)[["CompanyName", "CompanyNumber"]]
    .drop_duplicates()
    .head(API_LIMIT)
    .reset_index(drop=True)
)

print("Companies selected for API calls:", len(subset))
subset.head()

Companies selected for API calls: 1000


,CompanyName,CompanyNumber
0,SARAH RADCLYFFE PRODUCTIONS LIMITED,02739087
1,GEOVISTA LTD.,03620302
2,ANGLO SCHOOLS INTERNATIONAL SERVICES LIMITED,08536631
3,THE ENCORE GROUP (ENVELOPES AND PACKAGING) LIM...,01829336
4,HOOKEY & CO. LIMITED,00552187


In [17]:
BASE_URL = "https://api.company-information.service.gov.uk"

def ch_get(path, params=None):
    if not CH_API_KEY:
        raise ValueError("No Companies House API key found. Check .env for COMPANIES_HOUSE_API_KEY.")

    response = requests.get(
        BASE_URL + path,
        params=params,
        auth=HTTPBasicAuth(CH_API_KEY, ""),   # API key as username, blank password [1]
        timeout=10,
    )

    if response.status_code == 404:
        return None

    response.raise_for_status()
    return response.json()


def ch_get_all(path, per_page=100, max_items=1000):
    # The API defaults to 25 results per page, so we page through explicitly [1].
    results = []
    start = 0
    while True:
        data = ch_get(path, params={"items_per_page": per_page, "start_index": start})
        if data is None:
            break

        page = data.get("items", [])
        results.extend(page)

        # Officers API uses total_results; Filing History API uses total_count.
        total = data.get("total_results", data.get("total_count", len(results)))
        start += per_page

        if not page or start >= total or len(results) >= max_items:
            break

    return results

In [18]:
# Fetch officers for on company, loop through pages to get current and resigned officers
def fetch_company_officers(company_number):
    officers = []
    start_index = 0
    per_page = 100
    while True:
        data = ch_get(
            f"/company/{company_number}/officers",
            params={"items_per_page": per_page, "start_index": start_index}
        )

        if data is None:
            break

        page_items = data.get("items", [])
        officers.extend(page_items)

        total = data.get("total_results", len(officers))
        start_index += per_page

        if len(page_items) == 0 or start_index >= total:
            break

    return officers

Officer_cutoff = pd.Timestamp.today().normalize() -pd.DateOffset(months=18)
def summarise_officer_changes(company_number):
    active = 0
    appointed = 0
    resigned = 0
    for o in fetch_company_officers(company_number):
        is_dir = "director" in str(o.get("officer_role", "")).lower()
        a=pd.to_datetime(o.get("appointed_on"), errors="coerce")
        r=pd.to_datetime(o.get("resigned_on"), errors="coerce")
        if is_dir and pd.isna(r):
            active +=1
        if is_dir and pd.notna(a) and a>=Officer_cutoff:
            appointed +=1
        if is_dir and pd.notna(r) and r>=Officer_cutoff:
            resigned +=1
    return {
        "active_directors":active,
        "recent_director_appointments":appointed,
        "recent_director_resignations":resigned,
        "has_recent_director_change":(appointed>0) or (resigned >0)
    }

In [ ]:
Growth_filing_types = {"SH01": "Shares allotted (capital raise)", "SH06": "Share capital reduction"}

def fetch_company_filings(company_number, per_page=100, max_items=500):
    items, start = [], 0
    while True:
        data = ch_get(
            f"/company/{company_number}/filing-history",
            params={"items_per_page": per_page, "start_index": start},
        )
        if data is None:
            break
        page = data.get("items", [])
        items.extend(page)
        total = data.get("total_count", len(items))
        start += per_page
        if not page or start >= total or len(items) >= max_items:
            break
    return items

def summarise_filings(company_number):
    recent_growth = 0
    accounts_count = 0
    last_accounts_date = pd.NaT
    for f in fetch_company_filings(company_number):
        ftype = f.get("type")
        fcat = str(f.get("category", "")).lower()
        fdate = pd.to_datetime(f.get("date"), errors="coerce")
        if ftype in Growth_filing_types and pd.notna(fdate) and fdate >= Officer_cutoff:
            recent_growth += 1
        if fcat == "accounts":
            accounts_count += 1
            if pd.notna(fdate) and (pd.isna(last_accounts_date) or fdate > last_accounts_date):
                last_accounts_date = fdate
    return {
        "recent_growth_filings": recent_growth,
        "has_recent_growth_filing": recent_growth > 0,
        "accounts_filings_count": accounts_count,
        "last_accounts_filing_date": last_accounts_date,
    }

In [ ]:
enrich_rows = []

for i, (_, c) in enumerate(subset.iterrows(), start=1):
    cn, name = c["CompanyNumber"], c["CompanyName"]
    rec = {"CompanyName": name, "CompanyNumber": cn}

    try:
        rec.update(summarise_officer_changes(cn))
    except Exception:
        rec.update({
            "active_directors": np.nan,
            "recent_director_appointments": np.nan,
            "recent_director_resignations": np.nan,
            "has_recent_director_change": False,
        })

    try:
        rec.update(summarise_filings(cn))
    except Exception:
        rec.update({
            "recent_growth_filings": np.nan,
            "has_recent_growth_filing": False,
            "accounts_filings_count": np.nan,
            "last_accounts_filing_date": pd.NaT,
        })

    enrich_rows.append(rec)
    if i % 25 == 0:
        print(f"Processed {i}/{len(subset)} companies")

    time.sleep(0.5)

enrich_df = pd.DataFrame(enrich_rows)

out_path = r"C:\MSC\Project\lloyds-commercial-banking-intelligence-2026\data\processed\director_growth_enrichment.csv"
enrich_df.to_csv(out_path, index=False)
print(f"Saved {len(enrich_df)} rows → {out_path}")
enrich_df.head()

#### References

[1] Companies House Public Data API - Company Officers (list), incl. items, officer_role,
appointed_on, resigned_on, total_results, and authentication.
https://developer-specs.company-information.service.gov.uk/companies-house-public-data-api/reference/officers/list

[2] Companies House Public Data API - Filing history (list), incl. items, type, category,
date, and total_count pagination.
https://developer-specs.company-information.service.gov.uk/companies-house-public-data-api/reference/filing-history/list

[3] Companies House api-enumerations - filing type and description codes, including SH01
return of allotment of shares.
https://github.com/companieshouse/api-enumerations

#### Financial health check

Uses two data sources:
1. **Companies House charges API** — outstanding charges per company (secured lending signals).
2. **Accounts due-date columns** already in the base dataset — overdue accounts flag a late-filing risk.

A company is flagged as a financial health concern if any of:
- It has outstanding charges (potential existing secured creditors).
- Its accounts are overdue (NextDueDate < today).
- It has had no accounts filed in the last 24 months.

In [ ]:
def fetch_charges_summary(company_number):
    """Return outstanding and total charge counts from the CH charges API."""
    data = ch_get(f"/company/{company_number}/charges")
    if data is None:
        return {"charges_outstanding": 0, "charges_total": 0}
    items = data.get("items", [])
    outstanding = sum(
        1 for c in items
        if str(c.get("status", "")).lower() == "outstanding"
    )
    return {"charges_outstanding": outstanding, "charges_total": len(items)}


# Pull overdue-accounts flags from the base dataset columns we already have.
accounts_cols = ["CompanyNumber", "Accounts.NextDueDate", "Accounts.LastMadeUpDate", "Accounts.AccountCategory"]
accounts_base = (
    base[accounts_cols]
    .rename(columns={
        "Accounts.NextDueDate": "accounts_next_due",
        "Accounts.LastMadeUpDate": "accounts_last_made_up",
        "Accounts.AccountCategory": "accounts_category",
    })
    .copy()
)
accounts_base["accounts_next_due"] = pd.to_datetime(accounts_base["accounts_next_due"], errors="coerce")
accounts_base["accounts_last_made_up"] = pd.to_datetime(accounts_base["accounts_last_made_up"], errors="coerce")

today = pd.Timestamp.today().normalize()
stale_cutoff = today - pd.DateOffset(months=24)

accounts_base["accounts_overdue"] = accounts_base["accounts_next_due"] < today
accounts_base["accounts_stale"] = (
    accounts_base["accounts_last_made_up"].isna()
    | (accounts_base["accounts_last_made_up"] < stale_cutoff)
)

print("Overdue accounts:", accounts_base["accounts_overdue"].sum())
print("Stale accounts (>24 months):", accounts_base["accounts_stale"].sum())

In [ ]:
# Enrich the same subset with charges data and combine with accounts flags.
health_rows = []

for i, (_, c) in enumerate(subset.iterrows(), start=1):
    cn, name = c["CompanyNumber"], c["CompanyName"]
    rec = {"CompanyName": name, "CompanyNumber": cn}

    try:
        rec.update(fetch_charges_summary(cn))
    except Exception:
        rec.update({"charges_outstanding": np.nan, "charges_total": np.nan})

    health_rows.append(rec)
    if i % 25 == 0:
        print(f"  {i}/{len(subset)}")
    time.sleep(0.5)

health_df = pd.DataFrame(health_rows)

# Join in the accounts-due flags from the base dataset.
health_df = health_df.merge(
    accounts_base[["CompanyNumber", "accounts_overdue", "accounts_stale", "accounts_category"]],
    on="CompanyNumber",
    how="left",
)

# Composite financial health concern flag.
health_df["financial_health_concern"] = (
    (health_df["charges_outstanding"].fillna(0) > 0)
    | health_df["accounts_overdue"].fillna(False)
    | health_df["accounts_stale"].fillna(True)
)

out_health = r"C:\MSC\Project\lloyds-commercial-banking-intelligence-2026\data\processed\financial_health.csv"
health_df.to_csv(out_health, index=False)
print(f"\nSaved {len(health_df)} rows → {out_health}")
print("\nHealth concern breakdown:")
print(health_df["financial_health_concern"].value_counts())
health_df.head()